# HW3: Image Classification with CNN

Task: classify images with CNN models.

- Only competition data may be used.
- Models must be trained from scratch; no pretrained weights.
- Training images: `train/x_y.png`, where `x` is the category.
- Validation images: `valid/x_y.png`, where `x` is the category.
- Testing images: `test/n.png`, where `n` is the id.

## Download the Dataset

In [ ]:
# !pip install kagglehub==1.0.1

import inspect
import os
from pathlib import Path
from zipfile import ZipFile

DEFAULT_DOWNLOAD_DIR = Path("machine-learning/hung-yi-lee-machine-learning/HW3")
if not DEFAULT_DOWNLOAD_DIR.exists():
    DEFAULT_DOWNLOAD_DIR = Path.cwd()
DEFAULT_DOWNLOAD_DIR = DEFAULT_DOWNLOAD_DIR.resolve()
DATA_DOWNLOAD_DIR = DEFAULT_DOWNLOAD_DIR / "data"
DATA_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

# Optional: set this to a local dataset directory if the competition files were downloaded manually.
DATA_DOWNLOAD_ROOT = None
# DATA_DOWNLOAD_ROOT = "/path/to/ml2023spring-hw3"

# Optional: set this if the token is not under the current user's home directory.
KAGGLE_CREDENTIALS_DIR = None
# KAGGLE_CREDENTIALS_DIR = "/path/to/.kaggle"

KAGGLE_CONFIG_DIR = Path(
    KAGGLE_CREDENTIALS_DIR
    or os.environ.get("KAGGLE_CONFIG_DIR", Path.home() / ".kaggle")
).expanduser().resolve()
KAGGLE_ACCESS_TOKEN = KAGGLE_CONFIG_DIR / "access_token"
KAGGLE_JSON = KAGGLE_CONFIG_DIR / "kaggle.json"
os.environ["KAGGLE_CONFIG_DIR"] = str(KAGGLE_CONFIG_DIR)
os.environ["KAGGLEHUB_CACHE"] = str(DATA_DOWNLOAD_DIR)


def find_existing_data_dir(root: Path) -> Path | None:
    candidates = [root]
    if root.exists():
        candidates.extend(candidate for candidate in root.glob("**/*") if candidate.is_dir())

    for candidate in candidates:
        if (candidate / "train").is_dir() and (candidate / "valid").is_dir() and (candidate / "test").is_dir():
            return candidate.resolve()
    return None

def has_kaggle_credentials() -> bool:
    has_env_credentials = bool(os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"))
    has_access_token = KAGGLE_ACCESS_TOKEN.exists()
    has_json_credentials = KAGGLE_JSON.exists()
    if has_access_token:
        KAGGLE_ACCESS_TOKEN.chmod(0o600)
    if has_json_credentials:
        KAGGLE_JSON.chmod(0o600)
    return has_env_credentials or has_access_token or has_json_credentials


def download_competition(handle: str, output_dir: Path) -> str:
    import kagglehub

    signature = inspect.signature(kagglehub.competition_download)
    if "output_dir" in signature.parameters:
        return kagglehub.competition_download(handle, output_dir=str(output_dir))
    return kagglehub.competition_download(handle)


if DATA_DOWNLOAD_ROOT is None:
    existing_data_dir = find_existing_data_dir(DATA_DOWNLOAD_DIR)
    if existing_data_dir is not None:
        download_path = existing_data_dir
        path = str(download_path)
    else:
        if not has_kaggle_credentials():
            raise FileNotFoundError(
                f"Kaggle credentials were not found. Put the KaggleHub token at {KAGGLE_ACCESS_TOKEN}, "
                f"put legacy API credentials at {KAGGLE_JSON}, or set KAGGLE_USERNAME and KAGGLE_KEY "
                "before starting this notebook. Do not paste the token into this notebook."
            )

        try:
            downloaded_root = Path(download_competition("ml2023spring-hw3", DATA_DOWNLOAD_DIR)).expanduser().resolve()
        except Exception as error:
            error_name = error.__class__.__name__
            error_text = str(error)
            if error_name == "UnauthenticatedError":
                raise RuntimeError(
                    "KaggleHub still cannot authenticate. Make sure the current Jupyter kernel can see "
                    f"{KAGGLE_ACCESS_TOKEN} or {KAGGLE_JSON}, or restart the kernel after setting environment credentials. "
                    "Also join the competition and accept its rules on Kaggle before re-running this cell."
                ) from error
            if "403" in error_text and "rules" in error_text.lower():
                raise PermissionError(
                    "Kaggle authentication succeeded, but this account does not have permission to download "
                    "the competition data yet. Open https://www.kaggle.com/competitions/ml2023spring-hw3/rules, "
                    "join the competition, accept the rules, then re-run this cell."
                ) from error
            raise

        for zip_path in downloaded_root.glob("*.zip"):
            target_dir = downloaded_root / zip_path.stem
            if not target_dir.exists():
                with ZipFile(zip_path) as archive:
                    archive.extractall(target_dir)

        download_path = find_existing_data_dir(downloaded_root) or find_existing_data_dir(DATA_DOWNLOAD_DIR)
        if download_path is None:
            raise FileNotFoundError("Could not find the downloaded image data directory.")
        path = str(download_path)
else:
    download_path = Path(DATA_DOWNLOAD_ROOT).expanduser().resolve()
    path = str(download_path)

print("Data directory:", download_path)
print("KaggleHub cache:", os.environ["KAGGLEHUB_CACHE"])


## Setup and Model Overview

For image $X\in\mathbb{R}^{H\times W\times C}$, a CNN learns filters

$$
Y_{i,j,c}=\sigma\left(\sum_{m,n,d}X_{i+m,j+n,d}W_{m,n,d,c}+b_c\right).
$$

The classifier is trained from scratch by minimizing cross entropy:

$$
L=-\log p_y, \qquad p=\operatorname{softmax}(f_\theta(X)).
$$

In [2]:
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

try:
    import wandb
except ImportError:
    wandb = None


@dataclass
class Config:
    seed: int = 42
    image_size: int = 128
    batch_size: int = 64
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4
    max_epochs: int = 40
    early_stopping_patience: int = 8
    num_workers: int = 0
    gpu_id: int = 0
    model_filename: str = "cnn_image_classifier.pt"
    submission_filename: str = "submission_hw3.csv"
    use_wandb: bool = False
    wandb_project: str = "hung-yi-lee-machine-learning"
    wandb_run_name: str = "hw3-image-classification-cnn"


CONFIG = Config()


def set_seed(seed: int) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def choose_device() -> torch.device:
    if torch.cuda.is_available():
        gpu_count = torch.cuda.device_count()
        gpu_id = CONFIG.gpu_id if 0 <= CONFIG.gpu_id < gpu_count else 0
        if gpu_id != CONFIG.gpu_id:
            print(f"Requested CUDA GPU {CONFIG.gpu_id}, but only {gpu_count} GPU(s) are available. Falling back to cuda:0.")
        if hasattr(torch, "set_float32_matmul_precision"):
            torch.set_float32_matmul_precision("high")
        torch.backends.cudnn.benchmark = True
        device = torch.device(f"cuda:{gpu_id}")
        torch.cuda.set_device(device)
        print(f"Using CUDA GPU {gpu_id}: {torch.cuda.get_device_name(gpu_id)}")
        return device

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        print("Using Apple Metal Performance Shaders (MPS).")
        return torch.device("mps")

    print("Using CPU. GPU acceleration is not available in this environment.")
    return torch.device("cpu")


def to_device(tensor: torch.Tensor) -> torch.Tensor:
    return tensor.to(device, non_blocking=(device.type == "cuda"))


def dataloader_kwargs(shuffle: bool = False) -> dict:
    return {
        "batch_size": CONFIG.batch_size,
        "shuffle": shuffle,
        "num_workers": CONFIG.num_workers,
        "pin_memory": device.type == "cuda",
    }


def init_wandb():
    if not CONFIG.use_wandb:
        return None
    if wandb is None:
        raise ImportError("Install wandb or set CONFIG.use_wandb=False.")
    return wandb.init(
        project=CONFIG.wandb_project,
        name=CONFIG.wandb_run_name,
        config=asdict(CONFIG),
    )


set_seed(CONFIG.seed)
device = choose_device()
wandb_run = init_wandb()

print(f"Using device: {device}")
print(asdict(CONFIG))

Using device: mps
{'seed': 42, 'image_size': 128, 'batch_size': 64, 'learning_rate': 0.001, 'weight_decay': 0.0001, 'max_epochs': 40, 'early_stopping_patience': 8, 'num_workers': 0, 'model_filename': 'cnn_image_classifier.pt', 'submission_filename': 'submission_hw3.csv'}


## Locate and Inspect the Dataset

In [ ]:
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg"}


def find_data_dir() -> Path:
    roots: list[Path] = [Path.cwd(), Path.cwd().parent, Path(".").resolve()]
    if "path" in globals():
        roots.append(Path(path))

    candidates: list[Path] = []
    for root in roots:
        candidates.append(root)
        if root.exists():
            candidates.extend(candidate for candidate in root.glob("**/*") if candidate.is_dir())

    for candidate in candidates:
        if (candidate / "train").is_dir() and (candidate / "valid").is_dir() and (candidate / "test").is_dir():
            return candidate.resolve()

    raise FileNotFoundError("Could not find a directory containing train, valid, and test folders.")


def list_images(directory: Path) -> list[Path]:
    return sorted(path for path in directory.iterdir() if path.suffix.lower() in IMAGE_EXTENSIONS)


def label_from_path(path: Path) -> int:
    return int(path.stem.split("_", maxsplit=1)[0])


def test_id_from_path(path: Path) -> int:
    return int(path.stem)


DATA_DIR = find_data_dir()
TRAIN_DIR = DATA_DIR / "train"
VALID_DIR = DATA_DIR / "valid"
TEST_DIR = DATA_DIR / "test"

train_paths = list_images(TRAIN_DIR)
valid_paths = list_images(VALID_DIR)
test_paths = sorted(list_images(TEST_DIR), key=test_id_from_path)
class_ids = sorted({label_from_path(path) for path in train_paths + valid_paths})
class_to_index = {class_id: index for index, class_id in enumerate(class_ids)}
index_to_class = {index: class_id for class_id, index in class_to_index.items()}

print("Data directory:", DATA_DIR)
print("Train images:", len(train_paths))
print("Valid images:", len(valid_paths))
print("Test images:", len(test_paths))
print("Classes:", class_ids)

## Prepare Image Datasets

In [4]:
def image_to_tensor(image: Image.Image, image_size: int, training: bool = False) -> torch.Tensor:
    image = image.convert("RGB").resize((image_size, image_size))
    if training and np.random.rand() < 0.5:
        image = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)

    array = np.asarray(image, dtype=np.float32) / 255.0
    array = (array - np.array([0.485, 0.456, 0.406], dtype=np.float32)) / np.array([0.229, 0.224, 0.225], dtype=np.float32)
    return torch.from_numpy(array).permute(2, 0, 1)


class ImageClassificationDataset(Dataset):
    def __init__(self, image_paths: list[Path], training: bool = False, with_labels: bool = True) -> None:
        self.image_paths = image_paths
        self.training = training
        self.with_labels = with_labels

    def __len__(self) -> int:
        return len(self.image_paths)

    def __getitem__(self, index: int):
        image_path = self.image_paths[index]
        x = image_to_tensor(Image.open(image_path), CONFIG.image_size, self.training)

        if self.with_labels:
            y = class_to_index[label_from_path(image_path)]
            return x, torch.tensor(y, dtype=torch.long)

        return x, torch.tensor(test_id_from_path(image_path), dtype=torch.long)


train_dataset = ImageClassificationDataset(train_paths, training=True, with_labels=True)
valid_dataset = ImageClassificationDataset(valid_paths, training=False, with_labels=True)
test_dataset = ImageClassificationDataset(test_paths, training=False, with_labels=False)

train_loader = DataLoader(train_dataset, **dataloader_kwargs(shuffle=True))
valid_loader = DataLoader(valid_dataset, **dataloader_kwargs())
test_loader = DataLoader(test_dataset, **dataloader_kwargs())

## Build and Train the CNN

In [5]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int) -> None:
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Dropout2d(p=0.1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class SimpleCNN(nn.Module):
    def __init__(self, num_classes: int) -> None:
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3, 32),
            ConvBlock(32, 64),
            ConvBlock(64, 128),
            ConvBlock(128, 256),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.features(x))


model = SimpleCNN(num_classes=len(class_ids)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG.learning_rate, weight_decay=CONFIG.weight_decay)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)

trainable_params = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
print(f"Trainable parameters: {trainable_params:,}")
if wandb_run is not None:
    wandb.watch(model, log="gradients", log_freq=100)
model

Trainable parameters: 1,207,531


SimpleCNN(
  (features): Sequential(
    (0): ConvBlock(
      (block): Sequential(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU()
        (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (5): ReLU()
        (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
        (7): Dropout2d(p=0.1, inplace=False)
      )
    )
    (1): ConvBlock(
      (block): Sequential(
        (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU()
        (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (4): BatchNo

In [6]:
def run_epoch(model: nn.Module, loader: DataLoader, training: bool) -> tuple[float, float]:
    model.train(training)
    total_loss = 0.0
    correct = 0
    total = 0

    for x_batch, y_batch in tqdm(loader, leave=False):
        x_batch = to_device(x_batch)
        y_batch = to_device(y_batch)

        with torch.set_grad_enabled(training):
            logits = model(x_batch)
            loss = criterion(logits, y_batch)

        if training:
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * len(x_batch)
        correct += (logits.argmax(dim=1) == y_batch).sum().item()
        total += len(x_batch)

    return total_loss / total, correct / total


history: list[dict[str, float]] = []
best_valid_accuracy = 0.0
epochs_without_improvement = 0
checkpoint_path = Path(CONFIG.model_filename)

for epoch in range(1, CONFIG.max_epochs + 1):
    train_loss, train_accuracy = run_epoch(model, train_loader, training=True)
    valid_loss, valid_accuracy = run_epoch(model, valid_loader, training=False)
    scheduler.step(valid_accuracy)

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "valid_loss": valid_loss,
        "valid_accuracy": valid_accuracy,
    })

    if wandb_run is not None:
        wandb.log({
            "epoch": epoch,
            "train/loss": train_loss,
            "train/accuracy": train_accuracy,
            "valid/loss": valid_loss,
            "valid/accuracy": valid_accuracy,
            "learning_rate": optimizer.param_groups[0]["lr"],
        })

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} train_acc={train_accuracy:.4f} | "
        f"valid_loss={valid_loss:.4f} valid_acc={valid_accuracy:.4f}"
    )

    if valid_accuracy > best_valid_accuracy:
        best_valid_accuracy = valid_accuracy
        epochs_without_improvement = 0
        torch.save({
            "model_state_dict": model.state_dict(),
            "config": asdict(CONFIG),
            "class_ids": class_ids,
            "valid_accuracy": valid_accuracy,
        }, checkpoint_path)
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= CONFIG.early_stopping_patience:
        print(f"Early stopping at epoch {epoch}.")
        break

checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
print(f"Best validation accuracy: {checkpoint['valid_accuracy']:.4f}")
if wandb_run is not None:
    wandb.summary["best_valid_accuracy"] = checkpoint["valid_accuracy"]
    wandb.finish()

  0%|          | 0/157 [00:00<?, ?it/s]

  0%|          | 0/57 [00:00<?, ?it/s]

Epoch 01 | train_loss=2.0779 train_acc=0.2660 | valid_loss=1.9474 valid_acc=0.3184


  0%|          | 0/157 [00:00<?, ?it/s]

  0%|          | 0/57 [00:00<?, ?it/s]

...

  0%|          | 0/157 [00:00<?, ?it/s]

  0%|          | 0/57 [00:00<?, ?it/s]

Epoch 09 | train_loss=1.5040 train_acc=0.4891 | valid_loss=1.3847 valid_acc=0.5174


  0%|          | 0/157 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
history_df = pd.DataFrame(history)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_df["epoch"], history_df["train_loss"], label="train")
axes[0].plot(history_df["epoch"], history_df["valid_loss"], label="valid")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history_df["epoch"], history_df["train_accuracy"], label="train")
axes[1].plot(history_df["epoch"], history_df["valid_accuracy"], label="valid")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()
plt.show()

## Predict the Test Set and Export the Submission

In [ ]:
@torch.no_grad()
def predict(model: nn.Module, loader: DataLoader) -> pd.DataFrame:
    model.eval()
    rows: list[dict[str, int]] = []

    for x_batch, id_batch in tqdm(loader):
        logits = model(to_device(x_batch))
        predicted_indices = logits.argmax(dim=1).cpu().numpy()

        for image_id, predicted_index in zip(id_batch.numpy(), predicted_indices):
            rows.append({
                "Id": int(image_id),
                "Category": int(index_to_class[int(predicted_index)]),
            })

    return pd.DataFrame(rows).sort_values("Id").reset_index(drop=True)


submission = predict(model, test_loader)
submission_path = Path(CONFIG.submission_filename)
submission.to_csv(submission_path, index=False)

print(f"Saved submission to: {submission_path.resolve()}")
submission.head()